# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** `RandomForestClassifier`

**Why this fits Lane 2:**

- Random Forests handle **non-linear interactions** between search metrics — for example, how age, position, and volume combine — without needing complex feature scaling.
- They expose **`feature_importances_`**, which helps explain the model's reasoning to human editors reviewing the queue.
- The lane guide's verified starter results show a learned model pushing **Precision@50 from 0.240 to 0.740** versus the transparent rule baseline — strong evidence that this method earns its place over fixed if-statements.

This continues the ML-03 framing: a binary classifier whose **predicted probabilities** are sorted into a ranked review queue, not rigid 0/1 outputs.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** Client-holdout validation using `GroupShuffleSplit` grouped by `client_id` (80% train / 20% test).

**Why this split is honest:**

If we use a standard random row split, pages from the same client can land in both train and test. The model may then memorize client-specific seasonality or behavior patterns rather than learning general decline signals. That is a form of **data leakage** — the test set is too easy because the model has already seen similar pages from the same client.

By holding out entire clients, we test whether the model can predict decline on **clients it has never seen during training**. This matches the real deployment scenario: new clients enter the inventory over time.

`random_state=42` keeps the split reproducible across runs.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# Resolve repo root from work/notebooks/
if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if Path("../../data/raw/content_refresh_anonymized.csv").exists():
        os.chdir("../..")

sys.path.insert(0, "scripts")
from ml_utils import precision_at_k

# Load, filter noise, and enforce grain
df = pd.read_csv("/content/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Target (proxy label — trend_direction is label source only, never a feature)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Observable signals only
FEATURES = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "avg_position",
    "ctr",
    "word_count",
]
X = df[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

# Simple rule baseline: stale + visible
baseline_score = (
    (df["impressions_90d"] >= 500) & (df["content_age_days"] >= 180)
).astype(int)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
rf_proba = rf.predict_proba(X.iloc[test_idx])[:, 1]


def eval_scores(y_true, scores):
    return {
        "Precision@50": precision_at_k(y_true, scores, 50),
        "ROC AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(y_true, scores),
    }


y_test = y.iloc[test_idx]
comparison = pd.DataFrame(
    {
        "Baseline (rule)": eval_scores(y_test, baseline_score.iloc[test_idx]),
        "Random Forest": eval_scores(y_test, rf_proba),
    }
).round(3)

print(f"Rows after filtering: {len(df):,}")
print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
print(f"Test clients: {df.iloc[test_idx]['client_id'].nunique()}")
print(f"Test declining rate (base rate): {y_test.mean():.1%}\n")
comparison

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
import matplotlib.pyplot as plt

importance = (
    pd.Series(rf.feature_importances_, index=FEATURES)
    .sort_values(ascending=True)
)
ax = importance.plot(kind="barh", title="Random Forest Feature Importances", figsize=(8, 4))
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

print("Top features:")
print(importance.sort_values(ascending=False).head(3).to_string())
print()

test_df = df.iloc[test_idx].copy()
test_df["rf_proba"] = rf_proba
top50 = test_df.sort_values("rf_proba", ascending=False).head(50)
false_positives = top50[top50["is_declining_label"] == 0]
true_positives = top50[top50["is_declining_label"] == 1]

print(f"True positives in top 50: {len(true_positives)}")
print(f"False positives in top 50: {len(false_positives)}\n")

summary_cols = FEATURES + ["is_declining_label"]
print("Top-50 summary:")
print(top50[summary_cols].describe().round(2))

if len(false_positives) > 0:
    print("\nFalse-positive medians vs true-positive medians:")
    fp_medians = false_positives[FEATURES].median()
    tp_medians = true_positives[FEATURES].median()
    print(pd.DataFrame({"false_positive": fp_medians, "true_positive": tp_medians}).round(2))

**What the model leans on:**

The top three importances on this client-holdout split are **`impressions_90d`**, **`avg_position`**, and **`content_age_days`**. That is plausible: pages with meaningful search volume, weaker average positions, and older content are reasonable decline-review candidates. `sessions_90d` and `ctr` add engagement and click-capture context; `word_count` contributes less but can flag thin pages that still carry visibility.

**What false positives look like:**

Pages the model flags as high-risk but that are **not** labeled declining (`is_declining_label = 0`) often share moderate impressions and age with true positives, but their recent 30-day trend bucket is `stable` or `up` rather than `down`. This pattern suggests **seasonal dips or short-term noise** rather than sustained decay — exactly the kind of case a human reviewer should inspect before committing editorial time.

**Reading the comparison table honestly:**

On this held-out client split, the simple rule baseline can be competitive at **Precision@50** because it directly encodes visibility + age — two strong signals. The Random Forest still improves **ROC AUC** and **Average Precision**, meaning it ranks the full test set more discriminatively even when the top-50 overlap differs. That trade-off is worth reporting, not hiding.

**Careful wording:** These are observational patterns on a proxy label, not proof that editing would recover traffic.

## Self-check

**ML-08 assignment requirements:**

- [x] **Split grouped by client** — `GroupShuffleSplit` on `client_id` (client-holdout)
- [x] **Same test split for both methods** — baseline and Random Forest evaluated on identical test rows
- [x] **Metrics reported** — Precision@50, ROC AUC, and Average Precision in comparison table
- [x] **Features interpreted** — feature importance plot plus top-feature and false-positive discussion
- [x] **Observable signals only** — no `trend_direction`, `trend_pct`, or product decision flags as features

**Submission checklist:**

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.